# 🚀 ECommerce-IA — Entraînement EfficientNet-B4 sur Google Colab

**Projet SAHELYS** — Fine-tuning EfficientNet-B4 pour la classification de 120 catégories de produits e-commerce.

| Paramètre | Valeur |
|---|---|
| Modèle | EfficientNet-B4 (timm) — 19.3M params |
| Dataset | Fashion Product Images (120 catégories, 720 images) |
| GPU | T4 16 Go (Colab free tier) |
| Epochs | 30 (early stopping patience=7) |
| Image size | 380×380 |
| Batch | 16 (optimisé GPU) |
| Optimizer | AdamW + CosineAnnealingLR |
| Loss | CrossEntropy + Label Smoothing 0.1 |
| Stratégie | Unfreeze progressif (3 phases) |
| Augmentation | RandomHorizontalFlip + ColorJitter + RandomRotation |
| Persistance | Google Drive (reprise après déconnexion) |

**Auteur** : BAWANA Théodore

---

### ⚠️ Avant de commencer
1. Menu **Exécution** → **Modifier le type d’exécution** → **GPU T4**
2. Cliquez **Exécuter tout** (Ctrl+F9)
3. Google Drive sera monté automatiquement pour sauvegarder les checkpoints
4. **Si la session se déconnecte** : relancez « Exécuter tout » — tout reprend où ça s’est arrêté !

## 1️⃣ Montage Google Drive & Vérification GPU

In [ ]:
# === MONTER GOOGLE DRIVE POUR LA PERSISTANCE ===
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Dossier de travail sur Google Drive
DRIVE_DIR = Path('/content/drive/MyDrive/ECommerce-IA')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

# Sous-dossiers
DRIVE_MODELS = DRIVE_DIR / 'models'
DRIVE_DATA   = DRIVE_DIR / 'data'
DRIVE_RESULTS = DRIVE_DIR / 'results'
for d in [DRIVE_MODELS, DRIVE_DATA, DRIVE_RESULTS]:
    d.mkdir(parents=True, exist_ok=True)

print(f'\u2705 Google Drive monte')
print(f'   Dossier projet : {DRIVE_DIR}')
print(f'   Modeles        : {DRIVE_MODELS}')
print(f'   Donnees        : {DRIVE_DATA}')
print(f'   Resultats      : {DRIVE_RESULTS}')

# Lister les fichiers existants (pour la reprise)
existing = list(DRIVE_MODELS.glob('*'))
if existing:
    print(f'\n\U0001f4c2 Fichiers existants dans Drive :')
    for f in sorted(existing):
        size_mb = f.stat().st_size / 1e6
        print(f'   {f.name} ({size_mb:.1f} Mo)')
else:
    print(f'\n\U0001f4c2 Aucun fichier existant -- premiere execution')

In [ ]:
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA disponible : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    # Compatible toutes versions PyTorch (total_memory ou total_mem)
    total_mem = getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)
    print(f'VRAM            : {total_mem / 1e9:.1f} Go')
else:
    print('\n\u26a0\ufe0f  AUCUN GPU detecte !')
    print('   Menu Execution -> Modifier le type d\'execution -> GPU T4')

In [ ]:
!pip install -q timm kaggle tqdm pillow

## 2️⃣ Téléchargement du Dataset depuis Kaggle

Les données sont sauvegardées sur Google Drive. Si elles existent déjà, cette étape est **skippée automatiquement**.

In [ ]:
import os
import json

# === CONFIGURATION KAGGLE ===
KAGGLE_USERNAME = 'tedrostovenim'
KAGGLE_KEY = 'f2f88d5157f788d0e2fc387a9aa3289e'

if KAGGLE_KEY:
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
        json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
    print(f'\u2705 Kaggle configure pour : {KAGGLE_USERNAME}')
else:
    kaggle_path = os.path.expanduser('~/.kaggle/kaggle.json')
    if os.path.exists(kaggle_path):
        print('\u2705 kaggle.json deja present')
    else:
        # Verifier si on a deja les donnees sur Drive
        if (DRIVE_DATA / 'class_mapping.json').exists():
            print('\u2705 Donnees deja presentes sur Google Drive -- Kaggle non necessaire')
        else:
            print('\u26a0\ufe0f  Veuillez remplir KAGGLE_KEY ci-dessus ou uploader kaggle.json')
            print('   Allez sur https://www.kaggle.com/settings -> API -> Create New Token')

In [ ]:
%%time
import os
from pathlib import Path
import shutil
import zipfile

DATA_DIR = Path('/content/data')

# === VERIFIER SI LES DONNEES EXISTENT DEJA SUR DRIVE ===
splits_on_drive = DRIVE_DATA / 'splits' / 'train'
if splits_on_drive.exists() and len(list(splits_on_drive.iterdir())) >= 100:
    print('\u2705 Donnees deja organisees sur Google Drive -- SKIP du telechargement')
    n_cats = len(list(splits_on_drive.iterdir()))
    print(f'   {n_cats} categories trouvees dans {splits_on_drive}')
    
    # Copier depuis Drive vers /content pour la vitesse
    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)
    shutil.copytree(DRIVE_DATA, DATA_DIR)
    print(f'   Copie vers {DATA_DIR} (acces rapide)')

else:
    print('\U0001f4e5 Telechargement du dataset depuis Kaggle...')
    os.makedirs(DATA_DIR, exist_ok=True)
    
    # === VERIFIER QUE KAGGLE EST CONFIGURE ===
    kaggle_json = os.path.expanduser('~/.kaggle/kaggle.json')
    if not os.path.exists(kaggle_json):
        raise RuntimeError(
            '\u274c Kaggle non configure !\n'
            '   -> Retournez a la cellule 7 (Configuration Kaggle)\n'
            '   -> Remplissez KAGGLE_KEY avec votre cle API\n'
            '   -> Ou uploadez kaggle.json : https://www.kaggle.com/settings -> API -> Create New Token'
        )
    
    # Telecharger les images (fashion-product-images-small)
    !kaggle datasets download -d paramaggarwal/fashion-product-images-small -p {DATA_DIR} --unzip
    
    # Verifier que des fichiers ont ete telecharges
    downloaded = list(DATA_DIR.iterdir())
    if not downloaded:
        raise RuntimeError(
            '\u274c Le telechargement Kaggle a echoue (dossier vide) !\n'
            '   Verifiez votre cle API Kaggle.\n'
            '   -> https://www.kaggle.com/settings -> API -> Create New Token\n'
            f'   -> kaggle.json utilise : {kaggle_json}'
        )
    print(f'   \u2705 {len(downloaded)} elements telecharges')
    
    # Telecharger styles.csv separement (depuis le dataset complet)
    print('\n\U0001f4e5 Telechargement de styles.csv...')
    !kaggle datasets download -d paramaggarwal/fashion-product-images-dataset -f styles.csv -p {DATA_DIR}
    
    # Dezipper styles.csv.zip si present
    styles_zip = DATA_DIR / 'styles.csv.zip'
    if styles_zip.exists():
        with zipfile.ZipFile(str(styles_zip), 'r') as z:
            z.extractall(str(DATA_DIR))
        styles_zip.unlink()
        print('   styles.csv.zip extrait')
    
    # === RECHERCHE ROBUSTE de styles.csv ===
    styles_found = None
    # 1) Directement dans DATA_DIR
    if (DATA_DIR / 'styles.csv').exists():
        styles_found = DATA_DIR / 'styles.csv'
    else:
        # 2) Chercher recursivement dans tous les sous-dossiers
        results = list(DATA_DIR.rglob('styles.csv'))
        if results:
            styles_found = results[0]
            shutil.copy2(str(styles_found), str(DATA_DIR / 'styles.csv'))
            styles_found = DATA_DIR / 'styles.csv'
            print(f'   styles.csv trouve dans {results[0].parent} -> copie a la racine')
    
    if styles_found:
        size_mb = styles_found.stat().st_size / 1e6
        print(f'   \u2705 styles.csv ({size_mb:.1f} Mo)')
    else:
        # Dernier recours : telecharger tout le dataset complet
        print('   \u26a0\ufe0f styles.csv introuvable, telechargement du dataset complet...')
        !kaggle datasets download -d paramaggarwal/fashion-product-images-dataset -p /content/temp_dataset
        temp_dir = Path('/content/temp_dataset')
        # Chercher styles.csv dans les zips telecharges
        found_in_zip = False
        for zf in temp_dir.glob('*.zip'):
            with zipfile.ZipFile(str(zf), 'r') as z:
                for name in z.namelist():
                    if name.endswith('styles.csv'):
                        z.extract(name, str(DATA_DIR))
                        extracted = DATA_DIR / name
                        if extracted.parent != DATA_DIR:
                            shutil.move(str(extracted), str(DATA_DIR / 'styles.csv'))
                        print(f'   \u2705 styles.csv extrait depuis {zf.name}')
                        found_in_zip = True
                        break
            if found_in_zip:
                break
        if temp_dir.exists():
            shutil.rmtree(temp_dir)
        
        if not found_in_zip:
            # Lister le contenu pour debug
            print('\n\u274c Impossible de trouver styles.csv !')
            print(f'   Contenu de {DATA_DIR} :')
            for p in sorted(DATA_DIR.rglob('*'))[:30]:
                print(f'      {p}')
            raise FileNotFoundError('styles.csv introuvable apres tous les essais')
    
    # Chercher le dossier images
    IMAGES_DIR = DATA_DIR / 'images'
    if not IMAGES_DIR.exists() or len(list(IMAGES_DIR.glob('*.jpg'))) == 0:
        for d in DATA_DIR.rglob('*'):
            if d.is_dir() and len(list(d.glob('*.jpg'))) > 100:
                IMAGES_DIR = d
                break
    
    print(f'\n\U0001f4c1 Fichiers telecharges :')
    for item in sorted(os.listdir(str(DATA_DIR))):
        path = os.path.join(str(DATA_DIR), item)
        if os.path.isdir(path):
            count = len(os.listdir(path))
            print(f'   \U0001f4c2 {item}/ ({count} fichiers)')
        else:
            size_mb = os.path.getsize(path) / 1e6
            print(f'   \U0001f4c4 {item} ({size_mb:.1f} Mo)')

## 3️⃣ Organisation des données (120 catégories × 6 images)

Si les splits existent déjà → **skip automatique**.

In [ ]:
import pandas as pd
import shutil
from pathlib import Path
import random
import numpy as np
import json

random.seed(42)
np.random.seed(42)

DATA_DIR = Path('/content/data')
IMAGES_DIR = DATA_DIR / 'images'
RAW_DIR = DATA_DIR / 'raw'
SPLITS_DIR = DATA_DIR / 'splits'

# === VERIFIER SI DEJA FAIT ===
if (SPLITS_DIR / 'train').exists() and len(list((SPLITS_DIR / 'train').iterdir())) >= 100:
    n_train = sum(1 for _ in (SPLITS_DIR / 'train').rglob('*.jpg'))
    n_val = sum(1 for _ in (SPLITS_DIR / 'val').rglob('*.jpg'))
    n_test = sum(1 for _ in (SPLITS_DIR / 'test').rglob('*.jpg'))
    n_cats = len(list((SPLITS_DIR / 'train').iterdir()))
    print(f'\u2705 Donnees deja organisees -- SKIP')
    print(f'   Train : {n_train} images ({n_cats} categories)')
    print(f'   Val   : {n_val} images')
    print(f'   Test  : {n_test} images')

else:
    # === TROUVER styles.csv (peut etre dans un sous-dossier) ===
    styles_path = DATA_DIR / 'styles.csv'
    if not styles_path.exists():
        results = list(DATA_DIR.rglob('styles.csv'))
        if results:
            styles_path = results[0]
            print(f'\U0001f50d styles.csv trouve dans : {styles_path}')
        else:
            raise FileNotFoundError(
                f'styles.csv introuvable dans {DATA_DIR}. '
                f'Contenu : {[p.name for p in DATA_DIR.iterdir()]}'
            )
    
    print(f'\U0001f4e5 Chargement de {styles_path}...')
    df = pd.read_csv(styles_path, on_bad_lines='skip')
    print(f'   {len(df)} produits charges')
    
    # === TROUVER le dossier d images ===
    if not IMAGES_DIR.exists() or len(list(IMAGES_DIR.glob('*.jpg'))) == 0:
        IMAGES_DIR = None
        # Chercher un dossier 'images' dans l'arborescence
        for d in DATA_DIR.rglob('images'):
            if d.is_dir() and len(list(d.glob('*.jpg'))) > 100:
                IMAGES_DIR = d
                break
        # Sinon chercher n'importe quel dossier avec beaucoup de jpg
        if IMAGES_DIR is None:
            for d in DATA_DIR.rglob('*'):
                if d.is_dir() and len(list(d.glob('*.jpg'))) > 100:
                    IMAGES_DIR = d
                    break
        if IMAGES_DIR is None:
            # Lister le contenu pour debug
            print('\u26a0\ufe0f Contenu de DATA_DIR :')
            for p in sorted(DATA_DIR.rglob('*'))[:50]:
                print(f'   {p}')
            raise FileNotFoundError(f'Aucun dossier d\'images trouve dans {DATA_DIR}')
    
    nb_images = len(list(IMAGES_DIR.glob('*.jpg')))
    print(f'\U0001f5bc\ufe0f  {nb_images} images dans {IMAGES_DIR}')
    
    # Selection 120 categories x 6 images
    NB_CATEGORIES = 120
    IMAGES_PAR_CAT = 6
    
    existing_ids = set()
    for img_file in IMAGES_DIR.glob('*.jpg'):
        try:
            existing_ids.add(int(img_file.stem))
        except ValueError:
            pass
    
    df_valid = df[df['id'].isin(existing_ids)].copy()
    cat_counts = df_valid['articleType'].value_counts()
    eligible = cat_counts[cat_counts >= IMAGES_PAR_CAT]
    selected_categories = eligible.head(NB_CATEGORIES).index.tolist()
    
    print(f'   {len(eligible)} categories eligibles (>= {IMAGES_PAR_CAT} images)')
    print(f'   {len(selected_categories)} categories selectionnees')
    
    selected_products = []
    for cat in selected_categories:
        sampled = df_valid[df_valid['articleType'] == cat].sample(n=IMAGES_PAR_CAT, random_state=42)
        selected_products.append(sampled)
    df_selected = pd.concat(selected_products, ignore_index=True)
    print(f'\U0001f4e6 {len(df_selected)} images selectionnees ({len(selected_categories)} categories)')
    
    # Organiser en dossiers
    if RAW_DIR.exists():
        shutil.rmtree(RAW_DIR)
    RAW_DIR.mkdir(parents=True)
    
    for _, row in df_selected.iterrows():
        cat = str(row['articleType']).strip()
        src = IMAGES_DIR / f"{row['id']}.jpg"
        if src.exists():
            cat_dir = RAW_DIR / cat
            cat_dir.mkdir(exist_ok=True)
            shutil.copy2(src, cat_dir / f"{row['id']}.jpg")
    
    cats = sorted([d.name for d in RAW_DIR.iterdir() if d.is_dir()])
    print(f'\u2705 {len(cats)} dossiers crees dans {RAW_DIR}')
    
    # Repartition Train/Val/Test
    if SPLITS_DIR.exists():
        shutil.rmtree(SPLITS_DIR)
    for split in ['train', 'val', 'test']:
        (SPLITS_DIR / split).mkdir(parents=True)
    
    split_stats = {'train': 0, 'val': 0, 'test': 0}
    for cat_dir in sorted(RAW_DIR.iterdir()):
        if not cat_dir.is_dir():
            continue
        images = sorted(list(cat_dir.glob('*.jpg')))
        random.shuffle(images)
        n = len(images)
        n_test = max(1, n // 6)
        n_val = max(1, n // 6)
        n_train = n - n_val - n_test
        splits_map = {
            'train': images[:n_train],
            'val': images[n_train:n_train + n_val],
            'test': images[n_train + n_val:]
        }
        for split_name, split_images in splits_map.items():
            dest_dir = SPLITS_DIR / split_name / cat_dir.name
            dest_dir.mkdir(parents=True, exist_ok=True)
            for img in split_images:
                shutil.copy2(img, dest_dir / img.name)
                split_stats[split_name] += 1
    
    print('\u2705 Repartition terminee :')
    for split, count in split_stats.items():
        print(f'   {split:5s} : {count} images')
    
    # Sauvegarder le class_mapping
    class_mapping = {cat: idx for idx, cat in enumerate(sorted(cats))}
    with open(DATA_DIR / 'class_mapping.json', 'w') as f:
        json.dump(class_mapping, f, indent=2)
    
    # === SAUVEGARDER SUR GOOGLE DRIVE ===
    print('\n\U0001f4be Sauvegarde sur Google Drive...')
    if (DRIVE_DATA / 'splits').exists():
        shutil.rmtree(DRIVE_DATA / 'splits')
    if (DRIVE_DATA / 'raw').exists():
        shutil.rmtree(DRIVE_DATA / 'raw')
    shutil.copytree(SPLITS_DIR, DRIVE_DATA / 'splits')
    shutil.copytree(RAW_DIR, DRIVE_DATA / 'raw')
    shutil.copy2(DATA_DIR / 'class_mapping.json', DRIVE_DATA / 'class_mapping.json')
    print('\u2705 Donnees sauvegardees sur Google Drive')

## 4️⃣ Dataset PyTorch & DataLoaders

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import GradScaler, autocast
import torchvision.transforms as T
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import timm
import numpy as np
import json
import time
import os

# ============================================
# Constantes
# ============================================
IMAGE_SIZE = 380
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = Path('/content/data')
SPLITS_DIR = DATA_DIR / 'splits'
RAW_DIR = DATA_DIR / 'raw'
MODELS_DIR = Path('/content/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Chemins Drive (persistance)
DRIVE_DIR = Path('/content/drive/MyDrive/ECommerce-IA')
DRIVE_MODELS  = DRIVE_DIR / 'models'
DRIVE_RESULTS = DRIVE_DIR / 'results'

print(f'\U0001f5a5\ufe0f  Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'   GPU : {torch.cuda.get_device_name(0)}')

In [ ]:
# ============================================
# Dataset PyTorch
# ============================================
class ProductDataset(Dataset):
    def __init__(self, root_dir, transform=None, class_mapping=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.extensions = {'.jpg', '.jpeg', '.png', '.webp'}
        
        if class_mapping:
            self.class_to_idx = class_mapping
        else:
            self.class_to_idx = {
                d.name: idx for idx, d in enumerate(
                    sorted([d for d in self.root_dir.iterdir() if d.is_dir()])
                )
            }
        
        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}
        self.num_classes = len(self.class_to_idx)
        
        self.images, self.labels = [], []
        for cat_name, cat_idx in self.class_to_idx.items():
            cat_dir = self.root_dir / cat_name
            if not cat_dir.exists():
                continue
            for img_path in sorted(cat_dir.iterdir()):
                if img_path.suffix.lower() in self.extensions:
                    self.images.append(img_path)
                    self.labels.append(cat_idx)
        print(f'   \U0001f4e6 {len(self)} images, {self.num_classes} classes depuis {self.root_dir.name}/')
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        try:
            image = Image.open(self.images[idx]).convert('RGB')
        except Exception:
            image = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), (0, 0, 0))
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]


# ============================================
# Transformations
# ============================================
train_transform = T.Compose([
    T.Resize((IMAGE_SIZE + 20, IMAGE_SIZE + 20)),
    T.RandomCrop(IMAGE_SIZE),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(15),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.RandomGrayscale(p=0.05),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    T.RandomErasing(p=0.1),
])

val_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.CenterCrop(IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Charger le class_mapping
with open(DATA_DIR / 'class_mapping.json') as f:
    class_mapping = json.load(f)

# Creer les datasets
print('\n\U0001f4e6 Chargement des datasets :')
train_dataset = ProductDataset(SPLITS_DIR / 'train', train_transform, class_mapping)
val_dataset   = ProductDataset(SPLITS_DIR / 'val', val_transform, class_mapping)
test_dataset  = ProductDataset(SPLITS_DIR / 'test', val_transform, class_mapping)

NUM_CLASSES = train_dataset.num_classes

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f'\n\u2705 DataLoaders prets :')
print(f'   Train : {len(train_dataset)} images -> {len(train_loader)} batches')
print(f'   Val   : {len(val_dataset)} images -> {len(val_loader)} batches')
print(f'   Test  : {len(test_dataset)} images -> {len(test_loader)} batches')
print(f'   Classes : {NUM_CLASSES}')

## 5️⃣ Modèle, Fonctions & Reprise de checkpoint

In [ ]:
# ============================================
# Creation du modele EfficientNet-B4
# ============================================
DROPOUT_RATE = 0.3

print('\U0001f3d7\ufe0f  Creation du modele EfficientNet-B4...')
model = timm.create_model(
    'efficientnet_b4',
    pretrained=True,
    num_classes=NUM_CLASSES,
    drop_rate=DROPOUT_RATE
)

in_features = model.classifier.in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=DROPOUT_RATE),
    nn.Linear(in_features, NUM_CLASSES)
)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f'   Parametres totaux : {total_params:,}')
print(f'   Classes           : {NUM_CLASSES}')

In [ ]:
# ============================================
# Unfreeze Progressif
# ============================================
def geler_backbone(model):
    for param in model.parameters():
        param.requires_grad = False
    for param in model.classifier.parameters():
        param.requires_grad = True
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'   \u2744\ufe0f  Backbone gele -- {n:,} params entrainables')

def degeler_derniers_blocs(model, nb_blocs=2):
    blocks = list(model.blocks)
    for block in blocks[-nb_blocs:]:
        for param in block.parameters():
            param.requires_grad = True
    for name, param in model.named_parameters():
        if 'conv_head' in name or 'bn2' in name:
            param.requires_grad = True
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'   \U0001f513 Derniers {nb_blocs} blocs degeles -- {n:,} params entrainables')

def degeler_tout(model):
    for param in model.parameters():
        param.requires_grad = True
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'   \U0001f513 Reseau entierement degele -- {n:,} params entrainables')

def appliquer_phase(model, epoch, config):
    """Applique la bonne phase d unfreeze pour l epoch donnee."""
    if epoch <= config['phase1_end']:
        geler_backbone(model)
    elif epoch <= config['phase2_end']:
        geler_backbone(model)
        degeler_derniers_blocs(model, nb_blocs=2)
    else:
        degeler_tout(model)


# ============================================
# Label Smoothing CrossEntropy
# ============================================
class LabelSmoothingCE(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        self.confidence = 1.0 - smoothing
    
    def forward(self, pred, target):
        log_probs = torch.nn.functional.log_softmax(pred, dim=-1)
        nll = -log_probs.gather(dim=-1, index=target.unsqueeze(1)).squeeze(1)
        smooth = -log_probs.mean(dim=-1)
        return (self.confidence * nll + self.smoothing * smooth).mean()


# ============================================
# Early Stopping
# ============================================
class EarlyStopping:
    def __init__(self, patience=7, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.should_stop = False
    
    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                print(f'   \u23f9\ufe0f Early stopping (patience={self.patience})')
        return self.should_stop


# ============================================
# Train / Validate
# ============================================
def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    loss_sum, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc='  Train', leave=False)
    for images, labels in pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item() * images.size(0)
        _, pred = outputs.max(1)
        total += labels.size(0)
        correct += pred.eq(labels).sum().item()
        pbar.set_postfix(loss=f'{loss.item():.4f}', acc=f'{100.*correct/total:.1f}%')
    return loss_sum / total, correct / total


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    loss_sum, correct1, correct5, total = 0.0, 0, 0, 0
    for images, labels in tqdm(loader, desc='  Val  ', leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss_sum += loss.item() * images.size(0)
        total += labels.size(0)
        _, p1 = outputs.max(1)
        correct1 += p1.eq(labels).sum().item()
        _, p5 = outputs.topk(min(5, outputs.size(1)), dim=1)
        correct5 += sum(labels[i].item() in p5[i].tolist() for i in range(labels.size(0)))
    return loss_sum / total, correct1 / total, correct5 / total

print('\u2705 Fonctions d\'entrainement definies')

## 6️⃣ Entraînement avec reprise automatique

Chaque epoch est sauvegardée sur **Google Drive**.  
Si la session Colab se déconnecte, relancez simplement **« Exécuter tout »** — l’entraînement reprend à la dernière epoch sauvegardée.

In [ ]:
# ============================================
# HYPERPARAMETRES
# ============================================
CONFIG = {
    'epochs': 30,
    'lr': 3e-4,
    'weight_decay': 1e-4,
    'label_smoothing': 0.1,
    'patience': 7,
    'phase1_end': 5,
    'phase2_end': 15,
}

# ============================================
# REPRISE DEPUIS CHECKPOINT (Google Drive)
# ============================================
CHECKPOINT_PATH = DRIVE_MODELS / 'checkpoint_last.pth'
BEST_PATH       = DRIVE_MODELS / 'efficientnet_b4_best.pth'
HISTORY_PATH    = DRIVE_MODELS / 'training_history_checkpoint.json'

start_epoch = 1
best_val_acc = 0.0
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': [], 'val_top5': [],
    'lr': []
}

# Verifier s il existe un checkpoint sur Drive
if CHECKPOINT_PATH.exists():
    print('\U0001f504 Checkpoint trouve sur Google Drive ! Reprise...')
    ckpt = torch.load(str(CHECKPOINT_PATH), map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_val_acc = ckpt.get('best_val_acc', 0.0)
    
    # Charger l historique
    if HISTORY_PATH.exists():
        with open(str(HISTORY_PATH)) as f:
            history = json.load(f)
    
    print(f'   \u2705 Reprise a l\'epoch {start_epoch}')
    print(f'   \u2705 Meilleure accuracy precedente : {best_val_acc*100:.2f}%')
    print(f'   \u2705 Epochs deja faites : {start_epoch - 1}')
    
    if start_epoch > CONFIG['epochs']:
        print(f'\n\u2705 Entrainement deja termine ({CONFIG["epochs"]} epochs) ! Passez aux cellules suivantes.')
else:
    print('\U0001f195 Aucun checkpoint -- demarrage depuis zero')

# ============================================
# BOUCLE D ENTRAINEMENT
# ============================================
if start_epoch <= CONFIG['epochs']:
    # Appliquer la bonne phase pour l epoch de reprise
    print(f'\n\U0001f4cc Configuration de la phase pour epoch {start_epoch}...')
    appliquer_phase(model, start_epoch, CONFIG)
    
    criterion = LabelSmoothingCE(CONFIG['label_smoothing'])
    
    # LR adapte selon la phase
    if start_epoch <= CONFIG['phase1_end']:
        current_lr = CONFIG['lr']
    elif start_epoch <= CONFIG['phase2_end']:
        current_lr = CONFIG['lr'] * 0.5
    else:
        current_lr = CONFIG['lr'] * 0.1
    
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=current_lr, weight_decay=CONFIG['weight_decay']
    )
    
    # Charger optimizer state si disponible
    if CHECKPOINT_PATH.exists():
        try:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
            print('   \u2705 Etat de l\'optimizer restaure')
        except Exception:
            print('   \u26a0\ufe0f Optimizer reinitialise (changement de phase)')
    
    scheduler = CosineAnnealingLR(
        optimizer, T_max=CONFIG['epochs'], eta_min=1e-6,
        last_epoch=start_epoch - 2 if start_epoch > 1 else -1
    )
    scaler = GradScaler()
    
    # Restaurer early stopping
    early_stopping = EarlyStopping(patience=CONFIG['patience'])
    if history['val_loss']:
        early_stopping.best_loss = min(history['val_loss'])
    
    print(f'\n\U0001f3cb\ufe0f Entrainement epochs {start_epoch} -> {CONFIG["epochs"]}')
    print(f'   LR actuel   : {current_lr}')
    print(f'   Best acc    : {best_val_acc*100:.2f}%')
    print('=' * 100)
    
    total_start = time.time()
    
    for epoch in range(start_epoch, CONFIG['epochs'] + 1):
        epoch_start = time.time()
        
        # === Changement de phase ===
        if epoch == CONFIG['phase1_end'] + 1:
            print(f'\n\U0001f4cc Phase 2 : Derniers 2 blocs')
            geler_backbone(model)
            degeler_derniers_blocs(model, nb_blocs=2)
            optimizer = optim.AdamW(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=CONFIG['lr'] * 0.5, weight_decay=CONFIG['weight_decay']
            )
        elif epoch == CONFIG['phase2_end'] + 1:
            print(f'\n\U0001f4cc Phase 3 : Reseau complet')
            degeler_tout(model)
            optimizer = optim.AdamW(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=CONFIG['lr'] * 0.1, weight_decay=CONFIG['weight_decay']
            )
        
        # Train
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler)
        val_loss, val_top1, val_top5 = validate(model, val_loader, criterion)
        scheduler.step()
        lr = optimizer.param_groups[0]['lr']
        elapsed = time.time() - epoch_start
        
        # Historique
        history['train_loss'].append(float(train_loss))
        history['train_acc'].append(float(train_acc))
        history['val_loss'].append(float(val_loss))
        history['val_acc'].append(float(val_top1))
        history['val_top5'].append(float(val_top5))
        history['lr'].append(float(lr))
        
        # Meilleur modele
        is_best = val_top1 > best_val_acc
        if is_best:
            best_val_acc = val_top1
            best_ckpt = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_accuracy': val_top1,
                'val_loss': val_loss,
                'config': CONFIG,
                'class_to_idx': class_mapping,
                'num_classes': NUM_CLASSES,
            }
            # Sauvegarder localement ET sur Drive
            torch.save(best_ckpt, str(MODELS_DIR / 'efficientnet_b4_best.pth'))
            torch.save(best_ckpt, str(BEST_PATH))
        
        # === CHECKPOINT SUR GOOGLE DRIVE (chaque epoch) ===
        ckpt_data = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_acc': best_val_acc,
            'val_loss': val_loss,
            'val_accuracy': val_top1,
            'config': CONFIG,
        }
        torch.save(ckpt_data, str(CHECKPOINT_PATH))
        
        # Sauvegarder l historique sur Drive
        with open(str(HISTORY_PATH), 'w') as f:
            json.dump(history, f, indent=2)
        
        # Log
        marker = ' \U0001f3c6 BEST' if is_best else ''
        print(
            f'Epoch {epoch:2d}/{CONFIG["epochs"]} | '
            f'Train Loss: {train_loss:.4f} Acc: {train_acc*100:5.1f}% | '
            f'Val Loss: {val_loss:.4f} Top1: {val_top1*100:5.1f}% Top5: {val_top5*100:5.1f}% | '
            f'LR: {lr:.6f} | {elapsed:.0f}s{marker} \U0001f4be Drive'
        )
        
        # Early stopping
        if early_stopping(val_loss):
            print(f'\n\u23f9\ufe0f Arret anticipe a l\'epoch {epoch}')
            break
    
    total_time = time.time() - total_start
    print('\n' + '=' * 100)
    print(f'\U0001f3c6 MEILLEURE ACCURACY : {best_val_acc*100:.2f}%')
    print(f'\u23f1\ufe0f  Temps total : {total_time/60:.1f} minutes')
    print(f'\U0001f4be Modele sauve sur Google Drive : {BEST_PATH}')
    print(f'\U0001f4be Checkpoint sauve sur Google Drive : {CHECKPOINT_PATH}')

else:
    print('\u2705 Entrainement deja complete ! Chargement du meilleur modele...')
    if BEST_PATH.exists():
        best_ckpt = torch.load(str(BEST_PATH), map_location=DEVICE)
        model.load_state_dict(best_ckpt['model_state_dict'])
        best_val_acc = best_ckpt['val_accuracy']
        print(f'   Best Val Accuracy : {best_val_acc*100:.2f}%')

## 7️⃣ Courbes d’entraînement

In [ ]:
import matplotlib.pyplot as plt

# Charger l historique depuis Drive si necessaire
if not history['train_loss'] and HISTORY_PATH.exists():
    with open(str(HISTORY_PATH)) as f:
        history = json.load(f)

if not history['train_loss']:
    print('\u26a0\ufe0f Aucun historique disponible')
else:
    epochs_range = range(1, len(history['train_loss']) + 1)
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    axes[0].plot(epochs_range, history['train_loss'], 'b-o', label='Train', markersize=4)
    axes[0].plot(epochs_range, history['val_loss'], 'r-o', label='Val', markersize=4)
    axes[0].set_title('Loss', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(epochs_range, [a*100 for a in history['train_acc']], 'b-o', label='Train', markersize=4)
    axes[1].plot(epochs_range, [a*100 for a in history['val_acc']], 'r-o', label='Val Top-1', markersize=4)
    axes[1].plot(epochs_range, [a*100 for a in history['val_top5']], 'g-s', label='Val Top-5', markersize=4)
    axes[1].set_title('Accuracy (%)', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    axes[2].plot(epochs_range, history['lr'], 'g-o', markersize=4)
    axes[2].set_title('Learning Rate', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Epoch')
    axes[2].set_yscale('log')
    axes[2].grid(True, alpha=0.3)
    
    for ax in axes:
        ax.axvline(x=CONFIG['phase1_end']+0.5, color='orange', linestyle='--', alpha=0.5)
        ax.axvline(x=CONFIG['phase2_end']+0.5, color='purple', linestyle='--', alpha=0.5)
    
    plt.suptitle(f'EfficientNet-B4 -- {NUM_CLASSES} classes -- Best: {best_val_acc*100:.1f}%', fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    # Sauvegarder sur Drive
    curves_path = DRIVE_RESULTS / 'training_curves.png'
    plt.savefig(str(curves_path), dpi=150, bbox_inches='tight')
    plt.savefig(str(MODELS_DIR / 'training_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\U0001f4be Courbes sauvees sur Drive : {curves_path}')

## 8️⃣ Évaluation sur le Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, top_k_accuracy_score
import seaborn as sns

# Charger le meilleur modele depuis Drive
print('\U0001f4e5 Chargement du meilleur modele...')
best_model_path = BEST_PATH if BEST_PATH.exists() else MODELS_DIR / 'efficientnet_b4_best.pth'
checkpoint = torch.load(str(best_model_path), map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'   Epoch {checkpoint["epoch"]} -- Val Accuracy: {checkpoint["val_accuracy"]*100:.2f}%')

# Predictions
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Test'):
        images = images.to(DEVICE)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = outputs.max(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

test_acc = (all_preds == all_labels).mean()
top5_acc = top_k_accuracy_score(all_labels, all_probs, k=min(5, NUM_CLASSES))

print(f'\n{"="*60}')
print(f'\U0001f3c6 RESULTATS SUR LE TEST SET')
print(f'{"="*60}')
print(f'   Top-1 Accuracy : {test_acc*100:.2f}%')
print(f'   Top-5 Accuracy : {top5_acc*100:.2f}%')
print(f'   Images test    : {len(all_labels)}')
print(f'{"="*60}')

In [ ]:
# Classification report
idx_to_class = {v: k for k, v in class_mapping.items()}
present_labels = sorted(set(all_labels.tolist()))
target_names = [idx_to_class.get(i, f'class_{i}') for i in present_labels]

report = classification_report(
    all_labels, all_preds,
    labels=present_labels,
    target_names=target_names,
    output_dict=True,
    zero_division=0
)

class_results = []
for cls_name in target_names:
    if cls_name in report:
        r = report[cls_name]
        class_results.append((cls_name, r['precision'], r['recall'], r['f1-score'], int(r['support'])))

class_results.sort(key=lambda x: x[3], reverse=True)

print(f'\n\U0001f4ca Top 20 classes (par F1-score) :')
print(f'{"Classe":<25s} {"Precision":>10s} {"Recall":>10s} {"F1":>10s} {"Support":>8s}')
print('-' * 68)
for name, p, r, f1, s in class_results[:20]:
    print(f'{name:<25s} {p*100:>9.1f}% {r*100:>9.1f}% {f1*100:>9.1f}% {s:>7d}')

print(f'\n   Macro avg F1    : {report["macro avg"]["f1-score"]*100:.2f}%')
print(f'   Weighted avg F1 : {report["weighted avg"]["f1-score"]*100:.2f}%')

In [ ]:
# Matrice de confusion (top 20 classes)
top20_labels = present_labels[:20]
top20_names = [idx_to_class.get(i, f'c{i}') for i in top20_labels]

mask = np.isin(all_labels, top20_labels)
if mask.sum() > 0:
    cm = confusion_matrix(all_labels[mask], all_preds[mask], labels=top20_labels)
    
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=top20_names, yticklabels=top20_names, ax=ax)
    ax.set_xlabel('Predit', fontsize=12)
    ax.set_ylabel('Vrai', fontsize=12)
    ax.set_title(f'Matrice de Confusion (Top 20) -- Accuracy: {test_acc*100:.1f}%',
                 fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    
    cm_path = DRIVE_RESULTS / 'confusion_matrix.png'
    plt.savefig(str(cm_path), dpi=150, bbox_inches='tight')
    plt.savefig(str(MODELS_DIR / 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\U0001f4be Matrice sauvee sur Drive : {cm_path}')

## 9️⃣ Extraction des embeddings visuels (recherche par image)

In [ ]:
import pickle

EMBEDDINGS_PATH = DRIVE_MODELS / 'product_embeddings.pkl'

# === VERIFIER SI DEJA FAIT ===
if EMBEDDINGS_PATH.exists():
    print('\u2705 Embeddings deja extraits sur Google Drive -- SKIP')
    with open(str(EMBEDDINGS_PATH), 'rb') as f:
        embeddings_data = pickle.load(f)
    print(f'   {len(embeddings_data["labels"])} embeddings, dim={embeddings_data["embedding_dim"]}')

else:
    print('\U0001f50d Extraction des embeddings visuels...')
    
    feature_extractor = timm.create_model(
        'efficientnet_b4', pretrained=False, num_classes=0
    )
    
    state_dict = checkpoint['model_state_dict']
    backbone_dict = {k: v for k, v in state_dict.items() if not k.startswith('classifier')}
    feature_extractor.load_state_dict(backbone_dict, strict=False)
    feature_extractor = feature_extractor.to(DEVICE)
    feature_extractor.eval()
    
    all_embeddings, all_embed_labels, all_embed_paths = [], [], []
    
    for cat_dir in sorted(RAW_DIR.iterdir()):
        if not cat_dir.is_dir():
            continue
        for img_path in sorted(cat_dir.glob('*.jpg')):
            try:
                img = Image.open(img_path).convert('RGB')
                img_tensor = val_transform(img).unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    feat = feature_extractor(img_tensor)
                all_embeddings.append(feat.cpu().numpy().flatten())
                all_embed_labels.append(cat_dir.name)
                all_embed_paths.append(img_path.name)
            except Exception as e:
                print(f'   \u26a0\ufe0f {img_path.name}: {e}')
    
    embeddings_array = np.array(all_embeddings)
    print(f'\n\u2705 {len(all_embeddings)} embeddings extraits (dim={embeddings_array.shape[1]})')
    
    embeddings_data = {
        'embeddings': embeddings_array,
        'labels': all_embed_labels,
        'paths': all_embed_paths,
        'embedding_dim': embeddings_array.shape[1],
    }
    
    # Sauvegarder sur Drive
    with open(str(EMBEDDINGS_PATH), 'wb') as f:
        pickle.dump(embeddings_data, f)
    # Copie locale aussi
    with open(str(MODELS_DIR / 'product_embeddings.pkl'), 'wb') as f:
        pickle.dump(embeddings_data, f)
    
    print(f'\U0001f4be Embeddings sauves sur Drive : {EMBEDDINGS_PATH}')

## \U0001f4be Sauvegarder tout sur Google Drive & Télécharger

In [ ]:
import shutil

# Sauvegarder l historique final
history_final = {
    k: [float(v) for v in vals] for k, vals in history.items()
}
history_final['best_val_accuracy'] = float(best_val_acc)
history_final['test_accuracy'] = float(test_acc)
history_final['test_top5_accuracy'] = float(top5_acc)
history_final['config'] = CONFIG
history_final['num_classes'] = NUM_CLASSES
history_final['total_train_images'] = len(train_dataset)

# Sur Drive
with open(str(DRIVE_MODELS / 'training_history.json'), 'w') as f:
    json.dump(history_final, f, indent=2)
with open(str(DRIVE_MODELS / 'class_mapping.json'), 'w') as f:
    json.dump(class_mapping, f, indent=2)

# Localement aussi
with open(str(MODELS_DIR / 'training_history.json'), 'w') as f:
    json.dump(history_final, f, indent=2)
with open(str(MODELS_DIR / 'class_mapping.json'), 'w') as f:
    json.dump(class_mapping, f, indent=2)

print('\U0001f4e6 Fichiers sur Google Drive :')
for f in sorted(DRIVE_MODELS.glob('*')):
    size_mb = f.stat().st_size / 1e6
    print(f'   {f.name} ({size_mb:.1f} Mo)')

print(f'\n\U0001f4e6 Fichiers resultats :')
for f in sorted(DRIVE_RESULTS.glob('*')):
    size_mb = f.stat().st_size / 1e6
    print(f'   {f.name} ({size_mb:.1f} Mo)')

In [ ]:
# === TELECHARGER L ARCHIVE ZIP ===

# Creer une archive de tous les modeles
zip_path = '/content/ECommerce_IA_models'
shutil.make_archive(zip_path, 'zip', str(DRIVE_MODELS))

zip_size = os.path.getsize(f'{zip_path}.zip') / 1e6
print(f'\U0001f4e5 Archive creee : {zip_path}.zip ({zip_size:.1f} Mo)')

# Telecharger
try:
    from google.colab import files
    print('\n\u2b07\ufe0f  Telechargement automatique...')
    files.download(f'{zip_path}.zip')
    print('\u2705 Telechargement lance !')
except Exception:
    print('\n\u2139\ufe0f Les fichiers sont deja sur Google Drive :')
    print(f'   {DRIVE_MODELS}')
    print('   Vous pouvez les telecharger depuis drive.google.com')

## \u2705 Récapitulatif & Instructions

### Fichiers générés (sur Google Drive `/MyDrive/ECommerce-IA/`) :

```
ECommerce-IA/
\u251c\u2500\u2500 models/
\u2502   \u251c\u2500\u2500 efficientnet_b4_best.pth   \u2190 Meilleur mod\u00e8le (~80 Mo)
\u2502   \u251c\u2500\u2500 checkpoint_last.pth        \u2190 Dernier checkpoint (reprise)
\u2502   \u251c\u2500\u2500 product_embeddings.pkl     \u2190 Embeddings pour recherche image
\u2502   \u251c\u2500\u2500 class_mapping.json         \u2190 Mapping classes
\u2502   \u2514\u2500\u2500 training_history.json      \u2190 Historique complet
\u251c\u2500\u2500 results/
\u2502   \u251c\u2500\u2500 training_curves.png        \u2190 Courbes loss/accuracy
\u2502   \u2514\u2500\u2500 confusion_matrix.png       \u2190 Matrice de confusion
\u2514\u2500\u2500 data/
    \u251c\u2500\u2500 splits/                    \u2190 Donn\u00e9es organis\u00e9es
    \u2514\u2500\u2500 class_mapping.json
```

### Pour utiliser dans votre projet local :
1. T\u00e9l\u00e9chargez depuis Google Drive ou l'archive ZIP
2. Copiez :
   ```
   efficientnet_b4_best.pth  \u2192  models/classification/
   product_embeddings.pkl    \u2192  models/classification/
   class_mapping.json        \u2192  models/classification/
   training_history.json     \u2192  models/classification/
   ```
3. Lancez l'API : `python -m api.main`

### En cas de d\u00e9connexion Colab :
Relancez simplement **\u00ab Ex\u00e9cuter tout \u00bb** (Ctrl+F9).  
Le notebook reprendra automatiquement :
- \u2705 Les donn\u00e9es sont r\u00e9cup\u00e9r\u00e9es depuis Drive (pas de re-t\u00e9l\u00e9chargement)
- \u2705 L'entra\u00eenement reprend \u00e0 la derni\u00e8re epoch sauvegard\u00e9e
- \u2705 Les embeddings ne sont pas recalcul\u00e9s s'ils existent d\u00e9j\u00e0

---

\U0001f389 **F\u00e9licitations !** Mod\u00e8le entra\u00een\u00e9 et pr\u00eat pour la production.